In [4]:
# Instala lo que falte en el MISMO Python que usa este kernel
import importlib.util, subprocess, sys

paquetes = {"torch": "torch>=2.6", "transformers": "transformers", "pandas": "pandas",
            "matplotlib": "matplotlib", "safetensors": "safetensors"}
faltan = [p for m, p in paquetes.items() if importlib.util.find_spec(m) is None]

print("Python del kernel:", sys.executable)
if faltan:
    print("Instalando:", ", ".join(faltan)a)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
    print("✓ Listo. Si falla el import, reinicia el kernel.")
else:
    print("✓ Todo instalado.")

Python del kernel: c:\Users\juand\anaconda3\envs\Py-11\python.exe
Instalando: torch>=2.6, transformers, safetensors
✓ Listo. Si falla el import, reinicia el kernel.


In [5]:
import math
import os
import random
import textwrap
import time
import warnings
from pathlib import Path
from threading import Thread

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import transformers
from IPython.display import Markdown, display
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
    TextIteratorStreamer,
    get_linear_schedule_with_warmup,
)

In [ ]:
# Configuración de warnings y logging
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)

# Configuración de semilla
SEED = 42
def fijar_semilla(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fijar_semilla()

# Configuración de dispositivo
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Usando GPU:", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Usando GPU Apple MPS")
else:
    device = torch.device("cpu")
    print("Usando CPU")

print(f"torch {torch.__version__} | transformers {transformers.__version__} | device {device}")

Usando CPU
torch 2.14.1+cpu | transformers 5.19.0 | device cpu


In [12]:
MODEL_BASE = "datificate/gpt2-small-spanish"

TAG_USUARIO = '### Usuario:'
TAG_BOT = '### MiniGPT:'
SYSTEM_PROMPT = f"""Eres un modelo de lenguaje llamado MiniGPT, basado en GPT-2, entrenado para responder preguntas y mantener conversaciones en español."""    

CAPAS_CONGELADAS = 6 if device == "cpu" else 0
EPOCHS = 10
LEARNING_RATE = 1e-4 if CAPAS_CONGELADAS else 5e-5
BATCH_SIZE = 4
MAX_LEN_ENTRENAMIENTO = 192

DIR_SALIDA = Path("modelo") / "miniGPT-es"

In [14]:
def cargar_modelo_base(nombres):
    ultimo_error = None
    for nombre in nombres:
        try:
            print(f"Descargando/cargando '{nombre}' ...")
            tok = AutoTokenizer.from_pretrained(nombre)
            mod = AutoModelForCausalLM.from_pretrained(nombre)
            return nombre, tok, mod
        except Exception as e:  # sin internet, repo movido, etc.
            print(f"  ✗ No se pudo cargar '{nombre}': {type(e).__name__}: {str(e)[:150]}")
            ultimo_error = e
    raise RuntimeError("No se pudo cargar ningún modelo base.") from ultimo_error

inicio = time.time()

MODELO_BASE, tokenizer, model = cargar_modelo_base([MODEL_BASE])

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Se asignó el token de padding al token EOS: {tokenizer.pad_token}")

model.to(device)
model.eval()  
print(f"Modelo base '{MODELO_BASE}' cargado en {time.time() - inicio:.2f} segundos.")

Could not reach the Hub ([WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto). Using cached commit hash for 'datificate/gpt2-small-spanish'.
'Server disconnected without sending a response.' thrown while requesting GET https://huggingface.co/api/models/datificate/gpt2-small-spanish/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].


Descargando/cargando 'datificate/gpt2-small-spanish' ...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Modelo base 'datificate/gpt2-small-spanish' cargado en 2.00 segundos.


In [19]:
cfg = model.config

n_parametros = sum(p.numel() for p in model.parameters())
print(f"El modelo tiene {n_parametros:,} parámetros.")

El modelo tiene 124,439,808 parámetros.


In [31]:
pregunta = "Como te llamas"

prompt = (
    f"{SYSTEM_PROMPT}\n\n"
    f"{TAG_USUARIO} {pregunta}\n"
    f"{TAG_BOT} "
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    salida = model.generate(
        **inputs,
        max_new_tokens=60,          # Límite de tokens de respuesta
        do_sample=True,             # Muestreo probabilístico
        temperature=0.7,            # Grado de creatividad / coherencia
        top_p=0.9,                  # Nucleus sampling
        repetition_penalty=1.2,     # Evita repeticiones de texto
        pad_token_id=tokenizer.eos_token_id
    )

texto_generado = tokenizer.decode(salida[0], skip_special_tokens=True)
respuesta = texto_generado.split(TAG_BOT)[-1].strip()
print(f"Pregunta: {pregunta}")
print(f"Respuesta: {respuesta}")

Pregunta: Como te llamas
Respuesta: _PLAT_AUON
y ## Usuarios: Un intérprete de tipo Unix con el sistema mp3 que es capaz de usar el formato MPEG-4 (para el usuario) a través del uso de la opción codice_1, o por medio de una interfaz web


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

model_id = "clibrain/Llama-2-7b-ft-instruct-es"

model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_id)

def create_instruction(instruction, input_data=None, context=None):
    sections = {
        "Instrucción": instruction,
        "Entrada": input_data,
        "Contexto": context,
    }

    system_prompt = "A continuación hay una instrucción que describe una tarea, junto con una entrada que proporciona más contexto. Escriba una respuesta que complete adecuadamente la solicitud.\n\n"
    prompt = system_prompt

    for title, content in sections.items():
        if content is not None:
            prompt += f"### {title}:\n{content}\n\n"

    prompt += "### Respuesta:\n"

    return prompt


def generate(
        instruction,
        input=None,
        context=None,
        max_new_tokens=128,
        temperature=0.1,
        top_p=0.75,
        top_k=40,
        num_beams=4,
        **kwargs
):
    
    prompt = create_instruction(instruction, input, context)
    print(prompt.replace("### Respuesta:\n", ""))
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to("cuda")
    attention_mask = inputs["attention_mask"].to("cuda")
    generation_config = GenerationConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        num_beams=num_beams,
        **kwargs,
    )
    with torch.no_grad():
        generation_output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=True,
            max_new_tokens=max_new_tokens,
            early_stopping=True
        )
    s = generation_output.sequences[0]
    output = tokenizer.decode(s)
    return output.split("### Respuesta:")[1].lstrip("\n")

instruction = "Dame una lista de lugares a visitar en España."
print(generate(instruction))


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]